# Comparing Three Graph Models for Lipophilicity
# [RANDOM SPLIT]
We compare three model families on the same dataset:

1. **GCN (scratch)**: custom dense adjacency, one-hot atom features, batch_size=1.
2. **PyG GCN**: sparse batching with PyG, OGB atom features, standard `GCNConv`.
3. **PyG GIN**: stronger message passing (GIN) with OGB features.

Why try all three?
- The scratch model tests a minimal baseline with your own GCN layers.
- The PyG GCN tests a standard, optimized GCN with richer atom features.
- GIN is often stronger for molecular properties and can leverage the same features.

Each model uses early stopping, a LR scheduler, dropout, hidden=128, and up to 200 epochs. We save the best model weights to disk for later reuse.

In [1]:
import os
import pandas as pd
import numpy as np
from rdkit import Chem
import torch
from torch import nn
from tqdm import tqdm

import helpers as hp

from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GCNConv, GINConv, global_mean_pool

from ogb.utils import smiles2graph as smiles2graph_ogb
from ogb.graphproppred.mol_encoder import AtomEncoder

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
import torch.nn.functional as F

In [2]:
# Reproducibility and output folder
torch.manual_seed(0)
np.random.seed(0)

MODEL_DIR = 'saved_models'
os.makedirs(MODEL_DIR, exist_ok=True)

In [3]:
# Load dataset
from tdc.single_pred import ADME
data = ADME(name='Lipophilicity_AstraZeneca')
split = data.get_split()

Found local copy...
Loading...
Done!


In [4]:
# Missing values, canonicalization, duplicates
split['train']['_source'] = 'train'
split['valid']['_source'] = 'valid'
split['test']['_source'] = 'test'
combined = pd.concat([split['train'], split['valid'], split['test']], ignore_index=True)

combined['Mol'] = combined.Drug.apply(Chem.MolFromSmiles)
combined = combined.dropna(subset=['Mol'])
combined['Canonical_SMILES'] = combined.Mol.apply(lambda x: Chem.MolToSmiles(x, canonical=True))
combined.drop_duplicates(subset='Canonical_SMILES', keep='first', inplace=True)

split['train'] = combined[combined['_source'] == 'train'].drop(columns=['_source'])
split['valid'] = combined[combined['_source'] == 'valid'].drop(columns=['_source'])
split['test'] = combined[combined['_source'] == 'test'].drop(columns=['_source'])

TARGET_COL = 'Y'
SMILES_COL = 'Canonical_SMILES'

print(f"Training set shape: {split['train'].shape}")
print(f"Validation set shape: {split['valid'].shape}")
print(f"Test set shape: {split['test'].shape}")
print(f"Target range: [{combined[TARGET_COL].min():.2f}, {combined[TARGET_COL].max():.2f}] log(mol/L)")

Training set shape: (2940, 5)
Validation set shape: (420, 5)
Test set shape: (840, 5)
Target range: [-1.50, 4.50] log(mol/L)


## Model 1: GCN from Scratch
Dense adjacency, one-hot atom features, batch_size=1.

In [5]:
train_ds = hp.GNNDataset(split['train'][SMILES_COL], split['train'][TARGET_COL])
valid_ds = hp.GNNDataset(split['valid'][SMILES_COL], split['valid'][TARGET_COL])
test_ds = hp.GNNDataset(split['test'][SMILES_COL], split['test'][TARGET_COL])

train_loader = hp.DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = hp.DataLoader(valid_ds, batch_size=1, shuffle=False)
test_loader = hp.DataLoader(test_ds, batch_size=1, shuffle=False)

print(f'Split — Train: {len(train_ds)}  Val: {len(valid_ds)}  Test: {len(test_ds)}')

Split — Train: 2940  Val: 420  Test: 840


In [6]:
class GCNModelDropout(nn.Module):
    def __init__(self, in_features, hidden, out_features=1, n_layers=2, dropout=0.2, pool_fn=torch.mean):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(hp.GCNLayer(in_features, hidden))
        for _ in range(n_layers - 1):
            self.convs.append(hp.GCNLayer(hidden, hidden))
        self.dropout = nn.Dropout(dropout)
        self.readout = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, out_features),
        )
        self.pool_fn = pool_fn

    def forward(self, X, A):
        h = X
        for conv in self.convs:
            h = conv(h, A)
            h = self.dropout(h)
        h_graph = self.pool_fn(h, dim=0)
        return self.readout(h_graph)

In [ ]:
# gcn = GCNModelDropout(in_features=54, hidden=128, out_features=1, n_layers=2, dropout=0.2)
# optimiser = torch.optim.Adam(gcn.parameters(), lr=1e-3, weight_decay=1e-5)
# criterion = torch.nn.MSELoss()
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=10)

# N_EPOCHS = 200
# train_losses, val_losses = [], []

# best_val = float('inf')
# best_state = None
# patience = 20
# patience_left = patience

# for epoch in range(1, N_EPOCHS + 1):
#     gcn.train()
#     running = 0.0
#     for X_b, A_b, y_b in train_loader:
#         X_b, A_b, y_b = X_b.squeeze(0), A_b.squeeze(0), y_b.squeeze(0)
#         optimiser.zero_grad()
#         pred = gcn(X_b, A_b).squeeze()
#         loss = criterion(pred, y_b)
#         loss.backward()
#         optimiser.step()
#         running += loss.item()
#     train_rmse = (running / len(train_loader)) ** 0.5
#     train_losses.append(train_rmse)

#     gcn.eval()
#     val_loss = 0.0
#     with torch.no_grad():
#         for X_b, A_b, y_b in val_loader:
#             X_b, A_b, y_b = X_b.squeeze(0), A_b.squeeze(0), y_b.squeeze(0)
#             pred = gcn(X_b, A_b).squeeze()
#             val_loss += criterion(pred, y_b).item()
#     val_rmse = (val_loss / len(val_loader)) ** 0.5
#     val_losses.append(val_rmse)
#     scheduler.step(val_rmse)

#     if val_rmse < best_val:
#         best_val = val_rmse
#         best_state = {k: v.detach().cpu().clone() for k, v in gcn.state_dict().items()}
#         patience_left = patience
#     else:
#         patience_left -= 1

#     if epoch % 10 == 0:
#         print(f'Epoch {epoch:3d}  Train RMSE: {train_rmse:.4f}  Val RMSE: {val_rmse:.4f}')

#     if patience_left == 0:
#         print(f'Early stopping at epoch {epoch}')
#         break

# if best_state is not None:
#     gcn.load_state_dict(best_state)
#     torch.save(gcn.state_dict(), os.path.join(MODEL_DIR, 'gcn_scratch_best.pt'))
#     print('Saved:', os.path.join(MODEL_DIR, 'gcn_scratch_best.pt'))

Epoch  10  Train RMSE: 1.0513  Val RMSE: 0.9948
Epoch  20  Train RMSE: 1.0227  Val RMSE: 0.9713
Epoch  30  Train RMSE: 1.0141  Val RMSE: 0.9448
Epoch  40  Train RMSE: 0.9965  Val RMSE: 0.9487
Epoch  50  Train RMSE: 0.9854  Val RMSE: 0.9051
Epoch  60  Train RMSE: 0.9628  Val RMSE: 0.8969
Epoch  70  Train RMSE: 0.9760  Val RMSE: 0.8949
Epoch  80  Train RMSE: 0.9454  Val RMSE: 0.8812
Epoch  90  Train RMSE: 0.9511  Val RMSE: 0.8824
Epoch 100  Train RMSE: 0.9331  Val RMSE: 0.8574
Epoch 110  Train RMSE: 0.9414  Val RMSE: 0.8615
Epoch 120  Train RMSE: 0.9026  Val RMSE: 0.8352
Epoch 130  Train RMSE: 0.9091  Val RMSE: 0.8503
Epoch 140  Train RMSE: 0.8818  Val RMSE: 0.8167
Epoch 150  Train RMSE: 0.8784  Val RMSE: 0.8349
Epoch 160  Train RMSE: 0.8868  Val RMSE: 0.8163
Epoch 170  Train RMSE: 0.8708  Val RMSE: 0.8106
Epoch 180  Train RMSE: 0.8746  Val RMSE: 0.8058
Epoch 190  Train RMSE: 0.8813  Val RMSE: 0.8183
Epoch 200  Train RMSE: 0.8740  Val RMSE: 0.8043
Saved: saved_models/gcn_scratch_best.pt


## Model 2: PyG GCN
Sparse batching with OGB atom features, `GCNConv`, and Lightning.

In [7]:
class LipophilicityGraphData(InMemoryDataset):
    def __init__(self, root, df, split_name, smiles_col='Canonical_SMILES', target_col='Y'):
        self.df = df[[smiles_col, target_col]].reset_index(drop=True).copy()
        self.smiles_col = smiles_col
        self.target_col = target_col
        self.split_name = split_name
        super().__init__(root)
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return [f'{self.split_name}.pt']

    def download(self):
        pass

    def process(self):
        data_list = []
        for _, row in tqdm(self.df.iterrows(), total=len(self.df), desc=f'Featurising {self.split_name}'):
            smi = row[self.smiles_col]
            y_val = float(row[self.target_col])
            g = smiles2graph_ogb(smi)
            data_list.append(
                Data(
                    x=torch.tensor(g['node_feat'], dtype=torch.long),
                    edge_index=torch.tensor(g['edge_index'], dtype=torch.long),
                    edge_attr=torch.tensor(g['edge_feat'], dtype=torch.long),
                    y=torch.tensor([y_val], dtype=torch.float),
                )
            )
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

train_pyg = LipophilicityGraphData('./lipophilicity_pyg_ogb', split['train'], 'train')
valid_pyg = LipophilicityGraphData('./lipophilicity_pyg_ogb', split['valid'], 'valid')
test_pyg = LipophilicityGraphData('./lipophilicity_pyg_ogb', split['test'], 'test')

train_pyg_loader = PyGDataLoader(train_pyg, batch_size=64, shuffle=True)
val_pyg_loader = PyGDataLoader(valid_pyg, batch_size=64, shuffle=False)
test_pyg_loader = PyGDataLoader(test_pyg, batch_size=64, shuffle=False)

print(f'Train: {len(train_pyg)}  Val: {len(valid_pyg)}  Test: {len(test_pyg)}')

Train: 2940  Val: 420  Test: 840


In [8]:
class LightningGCN(pl.LightningModule):
    def __init__(self, hidden=128, lr=1e-3, dropout=0.2):
        super().__init__()
        self.save_hyperparameters()
        self.atom_emb = AtomEncoder(emb_dim=hidden)
        self.conv1 = GCNConv(hidden, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.dropout = dropout
        self.readout = torch.nn.Sequential(
            torch.nn.Linear(hidden, hidden // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden // 2, 1),
        )

    def forward(self, x, edge_index, batch):
        x = self.atom_emb(x)
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.readout(x)

    def _step(self, batch, stage):
        y_hat = self(batch.x, batch.edge_index, batch.batch).squeeze(-1)
        y = batch.y.squeeze(-1)
        loss = F.mse_loss(y_hat, y)
        self.log(f'{stage}/rmse', loss.sqrt(), batch_size=batch.num_graphs, prog_bar=True, on_epoch=True)
        return loss

    def training_step(self, batch, _):
        return self._step(batch, 'train')

    def validation_step(self, batch, _):
        self._step(batch, 'val')

    def test_step(self, batch, _):
        self._step(batch, 'test')

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val/rmse'
            }
        }

In [ ]:
# gcn_lightning = LightningGCN(hidden=128, lr=1e-3, dropout=0.2)

# logger = CSVLogger('logs', name='gcn_experiment')
# early_stop = EarlyStopping(monitor='val/rmse', mode='min', patience=20)
# checkpoint_cb = ModelCheckpoint(
#     monitor='val/rmse',
#     mode='min',
#     save_top_k=1,
#     dirpath=MODEL_DIR,
#     filename='pyg_gcn_best'
# )
# trainer = pl.Trainer(
#     max_epochs=200,
#     logger=logger,
#     enable_progress_bar=True,
#     log_every_n_steps=5,
#     enable_checkpointing=True,
#     enable_model_summary=False,
#     callbacks=[early_stop, checkpoint_cb],
# )
# trainer.fit(gcn_lightning, train_pyg_loader, val_pyg_loader)

# if checkpoint_cb.best_model_path:
#     best_model = LightningGCN.load_from_checkpoint(checkpoint_cb.best_model_path)
#     torch.save(best_model.state_dict(), os.path.join(MODEL_DIR, 'pyg_gcn_best.pt'))
#     print('Saved:', os.path.join(MODEL_DIR, 'pyg_gcn_best.pt'))

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Saved: saved_models/pyg_gcn_best.pt


## Model 3: PyG GIN
GIN often performs well on molecular property prediction.

In [9]:
class LightningGIN(pl.LightningModule):
    def __init__(self, hidden=128, lr=1e-3, dropout=0.2):
        super().__init__()
        self.save_hyperparameters()
        self.atom_emb = AtomEncoder(emb_dim=hidden)
        self.mlp1 = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.mlp2 = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.conv1 = GINConv(self.mlp1)
        self.conv2 = GINConv(self.mlp2)
        self.dropout = dropout
        self.readout = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x, edge_index, batch):
        x = self.atom_emb(x)
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.readout(x)

    def _step(self, batch, stage):
        y_hat = self(batch.x, batch.edge_index, batch.batch).squeeze(-1)
        y = batch.y.squeeze(-1)
        loss = F.mse_loss(y_hat, y)
        self.log(f'{stage}/rmse', loss.sqrt(), batch_size=batch.num_graphs, prog_bar=True, on_epoch=True)
        return loss

    def training_step(self, batch, _):
        return self._step(batch, 'train')

    def validation_step(self, batch, _):
        self._step(batch, 'val')

    def test_step(self, batch, _):
        self._step(batch, 'test')

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val/rmse'
            }
        }

In [10]:
gin_lightning = LightningGIN(hidden=128, lr=1e-3, dropout=0.2)

logger_gin = CSVLogger('logs', name='gin_experiment')
early_stop_gin = EarlyStopping(monitor='val/rmse', mode='min', patience=20)
checkpoint_gin = ModelCheckpoint(
    monitor='val/rmse',
    mode='min',
    save_top_k=1,
    dirpath=MODEL_DIR,
    filename='pyg_gin_best'
)

trainer_gin = pl.Trainer(
    max_epochs=200,
    logger=logger_gin,
    enable_progress_bar=True,
    log_every_n_steps=5,
    enable_checkpointing=True,
    enable_model_summary=False,
    callbacks=[early_stop_gin, checkpoint_gin],
)
trainer_gin.fit(gin_lightning, train_pyg_loader, val_pyg_loader)

if checkpoint_gin.best_model_path:
    best_model_gin = LightningGIN.load_from_checkpoint(checkpoint_gin.best_model_path)
    torch.save(best_model_gin.state_dict(), os.path.join(MODEL_DIR, 'pyg_gin_best.pt'))
    print('Saved:', os.path.join(MODEL_DIR, 'pyg_gin_best.pt'))

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Saved: saved_models/pyg_gin_best.pt
